# <div align="center">🚀 Madar (مدار) - Cloud LoRA Fine-Tuning Studio</div>
### <div align="center">Train LLMs natively in Ring/Madar with RingTensor & GPU Acceleration</div>

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com) 
[![GitHub](https://img.shields.io/badge/GitHub-Madar-blue.svg)](https://github.com/azzeddine2017/Madar) 
[![RingTensor](https://img.shields.io/badge/Autograd-RingTensor%20C%20Kernels-green.svg)]() 
[![LoRA](https://img.shields.io/badge/Fine--Tuning-Low--Rank%20Adaptation-purple.svg)]()

---

Welcome to the **Madar LoRA Fine-Tuning Studio**.

This notebook trains and fine-tunes LLMs (Qwen2.5, LLaMA-3, BitNet) directly using **Madar's native training library** (`libraries/madar_ai/trainer/`) and **RingTensor autograd & AdamW kernels**.

### 🌟 Key Capabilities:
1. **Zero External Python ML Dependencies:** 100% pure Madar/Ring dialect with C-level RingTensor acceleration.
2. **Pre-Synthesized Dataset:** Includes 730 instruction-tuning pairs extracted directly from the official Ring language Sphinx documentation.
3. **GPU-Accelerated Backprop:** OpenCL Zero-Copy GEMM kernels with OpenMP multi-threading.
4. **Stand-Alone Checkpoints:** Exports LoRA matrix adapters (`.tensor` and `.json`) ready for inference and merging.

## ⚙️ Step 1: Hardware & Environment Verification

In [ ]:
# Check NVIDIA GPU acceleration
!nvidia-smi

# Check CPU SIMD flags
!lscpu | grep -E "Model name|avx2|fma"

## 🔨 Step 2: Build Madar & RingTensor with High-Performance Flags

In [ ]:
# Clone repo or build from workspace
!if [ ! -d "Madar" ]; then git clone https://github.com/azzeddine2017/Madar.git; fi
%cd Madar

# Compile Madar Core and RingTensor extension
!chmod +x bin/install.sh
!cd extensions/ringtensor && ./buildgcc.sh
!cd bin && gcc -O3 -mavx2 -mfma -pthread -shared -fPIC ../language/src/*.c -I../language/include -o ring.so
!echo "Madar Core and RingTensor compiled successfully."

## 📥 Step 3: Download Base Model & Inspect Dataset

In [ ]:
# Download base Qwen2.5 GGUF model
!mkdir -p models
!wget -c -q --show-progress -O models/qwen2.5-0.5b-instruct-q4_0.gguf https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main/qwen2.5-0.5b-instruct-q4_0.gguf

# Check dataset size
!wc -l data/ring_instruct_dataset.jsonl

## 🚀 Step 4: Register Interactive `%%madar` Magic

In [ ]:
from IPython.core.magic import register_cell_magic
import subprocess

@register_cell_magic
def madar(line, cell):
    with open("_temp_run.ring", "w", encoding="utf-8") as f:
        f.write(cell)
    res = subprocess.run(["./bin/ring", "--json-errors", "_temp_run.ring"], capture_output=True, text=True)
    print(res.stdout)
    if res.stderr:
        print(res.stderr)
print("%%madar cell magic registered successfully.")

## 🏋️ Step 5: Execute Full LoRA Fine-Tuning in Madar

In [ ]:
%%madar
load "madar_ai.ring"
load "madar_trainer.ring"

cModelPath = "models/qwen2.5-0.5b-instruct-q4_0.gguf"
cDataPath  = "data/ring_instruct_dataset.jsonl"

? "=== Starting Cloud LoRA Fine-Tuning Session ==="
oModel = model_open(cModelPath)
oModel.setThreads(4)
oModel.setGPU(true)

oDataset = new MadarInstructDataset(cDataPath, "chatml")
? f"Loaded Dataset: {oDataset.size()} Instruction-Response Pairs"

# LoRA Hyperparameters: Rank 16, Alpha 32, Learning Rate 0.0002
oTrainer = new MadarLoRATrainer(oModel, oDataset, 16, 32.0, 0.0002)
oTrainer.nLogInterval  = 25
oTrainer.nSaveInterval = 100
oTrainer.cOutputDir    = "adapters/cloud_ring_lora"

# Run 3 Complete Training Epochs
oTrainer.train(3)


## 📊 Step 6: Verify and Evaluate LoRA Fine-Tuning Results

In [ ]:
%%madar
load "madar_ai.ring"

? "=== Evaluating Model Output on Complex Ring Code Generation ==="
oModel = model_open("models/qwen2.5-0.5b-instruct-q4_0.gguf")
oModel.setTemperature(0.2)

cQuestion = "Write a complete Ring class implementing an LRU Cache with put and get methods."
cPrompt = oModel.formatChatPrompt(cQuestion, "You are an elite Ring programming expert.")
? oModel.generate(cPrompt, 300)
